# 🛒 ApexPricing AI — Cloud Model Training Pipeline (Google Colab)

This notebook runs the complete training pipeline for the **Monotone LightGBM Demand Forecaster** and **Log-Log Econometric Price Elasticity Estimator**.

### Core Methodological Enforcements:
- **Strict Law of Demand**: $\partial \hat{D}/\partial P \le 0$ enforced via LightGBM monotone constraints.
- **Zero-Leakage Time Split**: Time-ordered 80/20 train/validation split.
- **Econometric Elasticities**: Relative log-log ordinary least squares regression across electronic SKUs.
- **Direct Export**: Downloads serialized model `.joblib` for immediate drop-in deployment to FastAPI/Streamlit.

## 📦 Step 1: Environment Setup & Clone Repository
Clone the repository from GitHub and install the necessary dependencies.

In [ ]:
# Clone project repository
!git clone https://github.com/ElevateDynasty/Smart-Pricing-Engine.git
%cd Smart-Pricing-Engine

# Install core dependencies
!pip install -q -r requirements.txt

## 🚀 Step 2: Execute Cloud Training Pipeline
Runs `scripts/train_colab.py` to prepare dataset, train Monotone LightGBM, fit price elasticity, and export artifacts.

In [ ]:
!python scripts/train_colab.py

## 📊 Step 3: Interactive Evaluation & Demand Curve Simulation
Load the trained model and simulate the demand response curve across prices.

In [ ]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
from src.models.demand_forecaster import DemandForecaster
from src.optimization.lifecycle_optimizer import LifecycleOptimizer

# Load the newly saved model
artifact = joblib.load('models_saved/demand_forecaster_lightgbm.joblib')
print('Trained Model Metrics:', artifact.get('metrics', {}))

# Initialize forecaster
forecaster = DemandForecaster()
forecaster.model = artifact['model']
forecaster.feature_engineer = artifact['feature_engineer']

# Run Lifecycle Optimization on flagship showcase scenario (iPhone 17 clearance)
optimizer = LifecycleOptimizer(forecaster)
result = optimizer.optimize_sku(
    product_id='SKU_IPHONE_17_256',
    product_name='iPhone 17 (256GB)',
    current_price=82900,
    mrp=89900,
    unit_cost=68000,
    current_inventory=480,
    competitor_avg_price=79990,
    competitor_min_price=78900,
    days_to_successor=60
)

print(f'AI Recommended Price: Rs. {result.recommended_price:,.0f} (Status Quo: Rs. 82,900)')
print(f'Expected Daily Demand: {result.expected_daily_demand:.1f} units/day (Status Quo: {result.status_quo_daily_demand:.1f})')
print(f'Expected Profit Lift : +Rs. {result.profit_lift_amount:,.0f} (+{result.profit_lift_pct}%)')

# Plot Monotone Demand Response Curve
test_prices = np.linspace(72000, 89900, 50)
demands = [forecaster.predict_point(
    selling_price=p,
    mrp=89900,
    competitor_avg_price=79990,
    competitor_min_price=78900,
    unit_cost=68000,
    opening_inventory=480,
    days_to_successor=60
) for p in test_prices]

plt.figure(figsize=(9, 4.5))
plt.plot(test_prices, demands, color='#2563eb', lw=2.5, label='Monotone LightGBM Demand')
plt.axvline(result.recommended_price, color='#10b981', linestyle='--', lw=2, label=f'AI Optimal (Rs. {result.recommended_price:,.0f})')
plt.axvline(82900, color='#ef4444', linestyle=':', lw=2, label='Current Price (Rs. 82,900)')
plt.title('Demand Response Curve (Monotone Gradient Boosting)', fontsize=12, fontweight='bold')
plt.xlabel('Selling Price (INR)', fontsize=10)
plt.ylabel('Projected Units Sold / Day', fontsize=10)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 💾 Step 4: Download Trained Weights to Local Machine
Download the trained `.joblib` model file directly to your browser download folder.

In [ ]:
from google.colab import files
import os

model_file = 'models_saved/demand_forecaster_lightgbm.joblib'
if os.path.exists(model_file):
    print(f'Downloading {model_file} to your computer...')
    files.download(model_file)
else:
    print('Error: Model file does not exist yet. Please execute Step 2.')